# RAG01｜iPhone 知識庫：EmbeddingGemma 2 + FAISS
延續老師的 SentenceTransformer、Document/Query Template、正規化、IndexFlatIP 與四檔格式。**本作業修改**：公開 URL 下載、語意區段切分、Apple 產品 metadata。依序執行；GPU 非必要。

## 1. Colab 與安裝
Colab「執行階段→變更執行階段類型」可選 GPU。第一次需下載模型；CPU 可執行但較慢。

**重點：**請由上往下執行。RAG01 會從公開 GitHub URL 自動下載產品 ZIP，不需手動上傳。第一次建立索引需要下載 EmbeddingGemma 2；沒有 GPU 也能執行。


In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1" huggingface-hub numpy pandas torch

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib,json,re,tempfile,zipfile,io,requests
import numpy as np,pandas as pd,torch,faiss
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi
from IPython.display import display
MODEL_ID="google/embeddinggemma-2"
DOCUMENT_TEMPLATE="title: none | text: {text}"
QUERY_TEMPLATE="task: search result | query: {question}"
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.bfloat16 if DEVICE=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
BATCH_SIZE=4; TOKEN_LIMIT_REQUESTED=2048
# 修改為自己的 GitHub Raw ZIP 直連網址
DATA_URL="https://raw.githubusercontent.com/mlieschang1007/hw_7_RAG/main/data/iphone-documents.zip"
print(DEVICE,MODEL_ID)

## 2. 自動下載與 ZIP 驗證
將六份 TXT 打包為 iphone-documents.zip 並公開到 GitHub，填入 Raw URL。未換範例網址時明確停止。

**重點：**此 Notebook 已設定本作業 GitHub Raw URL。先確認 Repository 為 Public，並以無痕視窗確認網址可免登入下載。


In [ ]:
if "YOUR-USER" in DATA_URL: raise ValueError("請設定實際公開 DATA_URL")
r=requests.get(DATA_URL,timeout=60); r.raise_for_status()
if len(r.content)<100 or not r.content.startswith(b"PK\x03\x04"): raise ValueError("回應不是 ZIP")
source_sha256=hashlib.sha256(r.content).hexdigest(); DATA_DIR=Path(tempfile.mkdtemp())
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
 if z.testzip() is not None: raise ValueError("ZIP CRC error")
 ms=[m for m in z.infolist() if not m.is_dir() and m.filename.lower().endswith('.txt')]
 if not ms: raise ValueError("ZIP 無 TXT")
 for m in ms:
  dest=(DATA_DIR/m.filename).resolve()
  if not dest.is_relative_to(DATA_DIR.resolve()): raise ValueError("不安全 ZIP 路徑")
  dest.parent.mkdir(parents=True,exist_ok=True); dest.write_bytes(z.read(m))
txt_paths=sorted(DATA_DIR.rglob('*.txt')); print('bytes',len(r.content),'TXT',len(txt_paths),'SHA256',source_sha256)

## 3. 文件整理、metadata 與語意切分
保留老師 `doc_id/text/metadata` 格式及 Document Template。以標題區段切 chunk，來源與日期附在每筆 metadata；超出模型 token 預算會停止，不截斷。

In [ ]:
def parse_doc(path):
 raw=path.read_text(encoding='utf-8-sig').replace('\r\n','\n')
 def f(k):
  m=re.search(rf'^{re.escape(k)}[：:][ \t]*(.*)$',raw,re.M); return m.group(1).strip() if m else None
 title=f('標題') or path.stem; product=f('產品') or title; year=f('發表年份'); url=f('來源網址'); date=f('資料整理日期')
 return raw,title,product,year,url,date
documents=[]
for path in txt_paths:
 raw,title,product,year,url,date=parse_doc(path)
 if not url or not url.startswith('https://support.apple.com/'): raise ValueError(f'{path.name} 缺 Apple 官方 URL')
 body=re.split(r'^【資料限制】',raw,flags=re.M)[0]
 sections=re.split(r'(?=^【[^】]+】)',body,flags=re.M)
 for j,section in enumerate(sections):
  section=section.strip()
  if not section or section.startswith(('標題：','產品：','發表年份：','來源網址：','資料整理日期：','資料性質：')): continue
  h=re.match(r'^【([^】]+)】',section); cat=h.group(1) if h else '產品資訊'
 # 控制 chunk 約 1800 字，避免單份長文件截斷
  for n in range(0,len(section),1800):
   text=section[n:n+1800]; documents.append({'doc_id':f'{path.stem}_{j:02d}_{n//1800:02d}','text':f'{title}｜{cat}\n{text}','metadata':{'product':product,'year':year,'category':cat,'title':f'{title}｜{cat}','source_url':url,'date':date,'filename':path.name}})
if not documents: raise ValueError('沒有文件')
index_doc_ids=[d['doc_id'] for d in documents]
if len(set(index_doc_ids))!=len(index_doc_ids): raise ValueError('doc_id 重複')
display(pd.DataFrame([{'doc_id':d['doc_id'],'product':d['metadata']['product'],'category':d['metadata']['category'],'chars':len(d['text'])} for d in documents]))

## 4. EmbeddingGemma 2、向量正規化與 FAISS
記錄精確 Hugging Face commit。遇到模型存取或記憶體錯誤，不暗中替換模型。

**重點：**RAG01 與 RAG02 必須使用同一個模型 commit、Document/Query Template、向量維度與正規化設定；不要任意更換模型。


In [ ]:
MODEL_REVISION=HfApi().model_info(MODEL_ID,token=False).sha
model=SentenceTransformer(MODEL_ID,revision=MODEL_REVISION,token=False,device=DEVICE,config_kwargs={'vision_config':None,'audio_config':None},model_kwargs={'dtype':DTYPE,'attn_implementation':'eager'})
DIMENSION=int(model.get_sentence_embedding_dimension()); TOKEN_LIMIT=min(2048,int(model.max_seq_length))
inputs=[DOCUMENT_TEMPLATE.format(text=d['text']) for d in documents]
lengths=[len(model.tokenizer(x,add_special_tokens=True,truncation=False)['input_ids']) for x in inputs]
if max(lengths)>TOKEN_LIMIT: raise ValueError('有 chunk 超過 token 上限，調小切分長度')
emb=model.encode(inputs,prompt='',batch_size=BATCH_SIZE,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
emb=np.ascontiguousarray(emb,dtype=np.float32); faiss.normalize_L2(emb)
if emb.shape!=(len(documents),DIMENSION) or not np.isfinite(emb).all(): raise ValueError('向量驗證失敗')
index=faiss.IndexFlatIP(DIMENSION); index.add(emb)
assert index.ntotal==len(documents)==len(index_doc_ids)
print(type(index).__name__,index.ntotal,index.d,MODEL_REVISION)

## 5. iPhone 查詢試搜
相似度是排序值，不是正確率。先檢視命中的文字。

In [ ]:
def search_documents(question,top_k=3):
 q=QUERY_TEMPLATE.format(question=question.strip()); v=model.encode([q],prompt='',convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False); v=np.ascontiguousarray(v,dtype=np.float32); faiss.normalize_L2(v)
 scores,pos=index.search(v,min(top_k,index.ntotal))
 return [{'rank':i+1,'score':float(s),**documents[int(p)]} for i,(s,p) in enumerate(zip(scores[0],pos[0])) if p>=0]
QUESTION='iPhone 16 Pro 跟 iPhone 17 Pro 的螢幕及相機差異？'; results=search_documents(QUESTION,3)
display(pd.DataFrame([{'rank':x['rank'],'doc_id':x['doc_id'],'score':x['score'],'source':x['metadata']['source_url']} for x in results]))
for x in results: print(x['text'])

## 6. 輸出 rag-data.zip 並驗證
ZIP 根目錄含 index.faiss、documents.json、index_doc_ids.json、config.json。下載後上傳到 GitHub Release，RAG02 才能公開自動下載。

**繳交重點：**成功訊息與下載的 `rag-data.zip` 是 RAG01 的主要成果截圖。之後需把 ZIP 上傳到 GitHub Release，RAG02 才能自動下載。


In [ ]:
from google.colab import files
OUT=Path(tempfile.mkdtemp()); faiss.write_index(index,str(OUT/'index.faiss'))
(OUT/'documents.json').write_text(json.dumps(documents,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'index_doc_ids.json').write_text(json.dumps(index_doc_ids,ensure_ascii=False,indent=2),encoding='utf-8')
import importlib.metadata as pkg
config={'format_version':'1.0','created_at':datetime.now(ZoneInfo('Asia/Taipei')).isoformat(),'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'model_config_kwargs':{'vision_config':None,'audio_config':None},'dimension':DIMENSION,'normalize_embeddings':True,'index_type':'IndexFlatIP','metric':'cosine_similarity_via_normalized_inner_product','document_template':DOCUMENT_TEMPLATE,'query_template':QUERY_TEMPLATE,'encode_prompt':'','max_seq_length':TOKEN_LIMIT,'document_count':len(documents),'document_unit':'semantic_chunk_one_vector','source_archive_url':DATA_URL,'source_archive_sha256':source_sha256,'source_urls':sorted({d['metadata']['source_url'] for d in documents}),'package_versions':{n:pkg.version(n) for n in ['sentence-transformers','transformers','faiss-cpu','huggingface-hub','numpy','pandas','torch']},'file_sha256':{n:hashlib.sha256((OUT/n).read_bytes()).hexdigest() for n in ['index.faiss','documents.json','index_doc_ids.json']}}
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
ZIP=Path('/content/rag-data.zip')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for n in ['index.faiss','documents.json','index_doc_ids.json','config.json']: z.write(OUT/n,arcname=n)
with zipfile.ZipFile(ZIP) as z: assert z.testzip() is None and len(z.namelist())==4
assert faiss.read_index(str(OUT/'index.faiss')).ntotal==len(documents)==config['document_count']
for n,h in config['file_sha256'].items(): assert hashlib.sha256((OUT/n).read_bytes()).hexdigest()==h
print('ZIP/CRC/SHA256/index/document-count validation passed',ZIP.stat().st_size); files.download(str(ZIP))